# D37: FULL 90 câu bằng **ReAct + Calculator tool** (Qwen3-4B)

Dạng D37: $(a+bi)|z| = \dfrac{K}{z}+(a+bi)i$, mệnh đề nào đúng về
khoảng chứa $|z|$. Phương pháp: nhóm nhân tử $w=a+bi$, lấy mô-đun 2 vế,
đặt $t=|z|$, đặt $u=t^2$, giải phương trình bậc 2 theo $u$, chọn nghiệm
dương, rồi đối chiếu $t=\sqrt{u}$ với từng phương án. Bản chạy đầy đủ
của phương pháp đã kiểm chứng độc lập với **toàn bộ 91 câu D37** (khớp
91/91) trong `KLTN_D37_ReAct_Calculator_1cau.ipynb`.

Model **không tự tính tay bất kỳ phép nào**: mọi phép tính đều gọi tool
`Calculator` (sympy, chính xác tuyệt đối, **có bộ nhớ biến**) theo vòng
lặp ReAct thật:
`Thought → Action → Action Input → Observation → …`

Prompt được **ép mở đầu bằng `<think>\nThought:`** (forced prefix);
model không còn quyền tự chọn viết văn xuôi mở đầu trước khi vào định dạng
ReAct.

Vòng lặp cũng có **chốt chặn "Final Answer"**: nếu model đã viết xong
`Final Answer: \boxed{{<Letter>}}` mà không dừng sinh token sạch, harness
cắt và dừng ngay tại đó thay vì chạy tiếp toàn bộ lại lần 2.

## Điểm khác bản 1 câu: vòng lặp ReAct chạy THEO LÔ

Chạy tuần tự 90 câu sẽ mất hàng giờ. Ở đây mỗi **vòng** gọi vLLM **một lần**
cho tất cả các câu đang hoạt động (vLLM tự batching), rồi chạy Calculator
riêng cho từng câu, rồi generate tiếp.

Mỗi câu có **bộ nhớ biến riêng** (`MayTinh()` riêng), **ngân sách token
riêng**, và tự thoát khỏi lô khi viết xong `Final Answer`.

Hạn mức tool gọi/câu (`26          # quy trinh day du ~9 luot tinh (w,K,w_abs_sq,K_sq,u_sols,2x kiem tra dau,u_val,t_val) + toi da 4 luot doi chieu phuong an, cong du cho vai lan sua loi cu phap`) đủ dư cho khoảng 9 lượt tính
đầy đủ quy trình cộng tối đa 4 lượt so khớp phương án.

## Prompt & backend giống hệt bản 1 câu

Cell 3 (prompt + backend `MayTinh`, đặt trước khi `LLM(...)` khởi tạo để
an toàn với `multiprocessing.fork`) được **trích nguyên văn** từ notebook
1 câu bằng script `scratch/build_react_full_D37.py`; không gõ lại, nên
không có nguy cơ lệch giữa 2 bản.

## Trước khi chạy

Upload `plan_solve_prompts_D37.json` (90 câu, đã sinh sẵn từ
`Sinh_them_cau_hoi/So_phuc_day_du.csv`, đã loại câu gốc STT 8 nằm trong
few-shot) thành Kaggle Dataset (slug gợi ý `d37-full90`), gắn vào notebook,
bật GPU.

Kết quả: `/kaggle/working/d37_react_calculator_full90.csv`

In [ ]:
!pip install -q -U vllm
!pip uninstall -y -q torchcodec
import vllm; print('vLLM:', vllm.__version__)

In [ ]:
import os, re, json, time
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

DATA_PATH = '/kaggle/input/d37-full90/plan_solve_prompts_D37.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d37_react_calculator_full90.csv'

MODEL = 'Qwen/Qwen3-4B'
# Giu NGUYEN tham so sinh da kiem chung o ban 1 cau.
TEMPERATURE, TOP_P, TOP_K, SEED = 0.3, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
MAX_MODEL_LEN, MAX_NEW_TOKENS = 14336, 9216

AN_SO_TU_DO = {'u'}  # u = t^2 = |z|^2, an so trung gian can giai

# Dung 2 GPU neu co: gap doi KV cache va nhanh hon.
N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'So GPU: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(DATA_PATH, encoding='utf-8') as f:
    records = json.load(f)
print('So cau:', len(records))

# ---- Backend Calculator: dat O DAY, TRUOC khi LLM(...)/CUDA khoi tao ----
# (an toan multiprocessing.fork - xem giai thich trong comment ben duoi)
import sympy as sp
import multiprocessing as mp

GAN_BIEN_RE = re.compile(r'^([A-Za-z_]\w*)\s*=\s*(.+)$', re.S)
# Luoi an toan CHO CA BATCH: du da dung radsimp() de tranh treo may o hau
# het truong hop, sympy van khong dam bao toc do cho MOI to hop can thuc
# bat ky - chi can 1/90 cau roi vao truong hop xau la ca batch nghen theo
# (Calculator chay tuan tu tung cau). Dung TIEN TRINH CON that (co the bi
# giet cuong buc bang tin hieu he dieu hanh) thay vi thread: thread chi
# ngat duoc tai diem GIL duoc nhuong lai, KHONG dam bao neu tinh toan ket
# sau trong 1 loi goi C lien tuc (da xac nhan qua thuc te chay tren
# Kaggle). Pool tien trinh con nay duoc tao NGAY TAI DAY, TRUOC KHI
# LLM(...)/CUDA khoi tao ben duoi - vi vay an toan tuyet doi voi
# multiprocessing.fork (fork() SAU KHI CUDA da khoi tao moi la nguy hiem,
# tung gay treo o mot lan thu truoc do).
TIMEOUT_SECONDS = 20


def fast_simplify(expr):
    """Rut gon nhanh va ON DINH hon sp.simplify() thuan tuy.

    sp.simplify() la ham "thu tat ca chien luoc roi chon ket qua ngan nhat",
    rat cham (co the treo may) khi bieu thuc co nhieu MAU SO chua can bac
    hai khac goc (vd tu phep chia (Bx-Ay)/(Bx-Ax)) - dung sinh ra qua nhieu
    dang the hien khac nhau ma khong bao gio hop nhat lai. radsimp() giai
    quyet dung goc van de nay: no huu ti hoa mau so chua can NGAY LAP TUC,
    nen ket qua o moi buoc luon o dang gon, khong de cac mau can long tich
    luy qua tung phep tinh tiep theo. Dung radsimp() lam buoc rut gon CHINH
    (nhanh, gan nhu luon du); chi roi sang simplify() lam buoc du phong khi
    radsimp() chua dua duoc ve dang 0/dang gon nhat.
    """
    return sp.expand(sp.radsimp(expr))


def is_zero(expr):
    """Kiem tra bieu thuc co bang 0 khong.

    Buoc dau (radsimp) bat duoc phan lon truong hop that nhanh va tuyet
    doi chinh xac. Neu chua ket luan duoc, KHONG roi sang sp.simplify()
    (chung minh dai so - cham, khong dam bao toc do, day la duong tung
    gay treo may). Thay vao do, so sanh gia tri SO HOC voi do chinh xac
    rat cao (50 chu so thap phan) - dung nguyen tac may tinh Casio: so
    hai so thap phan thay vi chung minh dang thuc dai so. Voi mien bai
    toan nay (cac hang so dai so co dinh tu de bai, khong phai gia tri
    adversarial), sai so gan nhu khong the xay ra o do chinh xac nay.
    """
    rut_gon = sp.radsimp(expr)
    if rut_gon == 0:
        return True
    return abs(complex(rut_gon.evalf(50))) < 1e-40


class MayTinh:
    """May tinh sympy co bo nho bien (reset moi cau).

    an_so_tu_do: tap ten bien duoc PHEP giu tu do trong ket qua (khong bao
    loi 'undefined') du chua tung duoc gan gia tri - dung cho cac dang co
    an so chua biet can giai (vi du D35 dung x, y la an can tim, chi tro
    thanh so cu the sau buoc solve() o gan cuoi)."""

    def __init__(self, an_so_tu_do=()):
        self.an_so_tu_do = set(an_so_tu_do)
        # DANG KY SAN moi an so tu do la ky hieu SO THUC (real=True) ngay tu
        # dau - neu de sympify tu tao ky hieu (khi gap ten lan dau trong 1
        # bieu thuc), no se KHONG mac dinh la so thuc, khien Abs(t-4+I) khong
        # tu rut gon duoc thanh (t-4)**2+1 ma giu nguyen dang chua rut gon.
        self.ns = {ten: sp.Symbol(ten, real=True) for ten in self.an_so_tu_do}

    def _kiem_tra_ten_la(self, bieu_thuc):
        """sympify AM THAM bien ten chua dinh nghia thanh Symbol rong, VA
        ten HAM chua dinh nghia thanh mot AppliedUndef chua tinh (vd goi
        "evalf(...)" - khong phai ham that) - ca 2 deu khong bao loi, ket
        qua se vo nghia hoac khong tien trien ma model khong he hay biet.
        Chan lai moi ten bien KHONG nam trong bo nho VA khong nam trong
        danh sach an so tu do duoc khai bao truoc, VA moi loi goi ham chua
        duoc dinh nghia."""
        con_lai = getattr(bieu_thuc, 'free_symbols', set())
        ten_thieu = {str(x) for x in con_lai} - self.an_so_tu_do
        ham_la = {str(h.func) for h in getattr(bieu_thuc, 'atoms', lambda *a: set())(sp.core.function.AppliedUndef)}
        if not ten_thieu and not ham_la:
            return None
        if ham_la:
            ham_la_str = ', '.join(sorted(ham_la))
            return (f'undefined function name(s): {ham_la_str}. This is not a '
                    'recognized Calculator function. Only use the functions '
                    'documented in the tool instructions (sqrt, Abs, conjugate, '
                    'solve, Eq, nroots, re, im, expand, and ordinary +-*/**); do '
                    'not invent or guess a function name that is not listed there.')
        ten_la = ', '.join(sorted(ten_thieu))
        da_co = ', '.join(sorted(self.ns)) or '(chua co bien nao)'
        an_cho_phep = ', '.join(sorted(self.an_so_tu_do)) or '(khong co)'
        return (f'undefined name(s): {ten_la}. You used a name that is not in '
                f'calculator memory and is not a declared free unknown. Names '
                f'currently in memory: {da_co}. Declared free unknowns (allowed '
                f'without prior assignment): {an_cho_phep}. Either compute and '
                'store that name first (with "name = expression"), or rewrite '
                'the expression without it.')

    def tinh(self, bieu_thuc: str):
        """Tra ve (result_str, loi_str). Loi thi result_str = None."""
        s = bieu_thuc.strip()
        if not s:
            return None, 'empty expression'
        if '==' in s:
            return None, ('"==" is not supported. To compare two expressions '
                          'exactly, write Eq(left, right) instead.')

        ten = None
        m = GAN_BIEN_RE.match(s)
        # "Eq(a, b)" cung khop regex tren neu viet la "x = Eq(...)"; con
        # "Eq(a,b)" thuan thi khong co dau "=" o cap ngoai nen khong khop.
        if m and not s.lstrip().startswith('Eq('):
            ten, s = m.group(1), m.group(2)

        try:
            # 'Eq' duoc thay bang phien ban evaluate=False: sp.Eq() mac dinh
            # TU DONG thu kiem tra 2 ve co bang nhau NGAY LUC KHOI TAO (co che
            # rieng cua sympy, khac hoan toan ham is_zero() tu viet ben duoi) -
            # voi bieu thuc can long phuc tap, chinh buoc TU DONG nay co the
            # treo may, va treo TRUOC CA KHI chay_co_timeout kip can thiep (vi
            # no xay ra ngay trong luc sympify dang parse chuoi). Dung ban
            # evaluate=False de hoan toan doi viec so khop cho ham is_zero() -
            # da duoc kiem chung nhanh va dang tin cay - dam nhiem.
            ns_de_parse = dict(self.ns)
            ns_de_parse['Eq'] = lambda a, b: sp.Eq(a, b, evaluate=False)
            parsed = sp.sympify(s, locals=ns_de_parse)
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

        try:
            if isinstance(parsed, sp.Equality):
                loi_ten = self._kiem_tra_ten_la(parsed.lhs - parsed.rhs)
                if loi_ten:
                    return None, loi_ten
                bang_nhau, loi_timeout = chay_co_timeout(is_zero, parsed.lhs - parsed.rhs)
                if loi_timeout:
                    return None, loi_timeout
                ket_qua_bool = sp.true if bang_nhau else sp.false
                if ten:
                    self.ns[ten] = ket_qua_bool
                    return f'{ten} = {ket_qua_bool}', None
                return ('True' if bang_nhau else 'False'), None

            if isinstance(parsed, (list, tuple, sp.FiniteSet)):
                # Ket qua tra ve tu solve(...): danh sach nghiem. Khong goi
                # expand/simplify tren list - kiem tra tung phan tu rieng.
                ds = list(parsed)
                for phan_tu in ds:
                    loi_ten = self._kiem_tra_ten_la(phan_tu)
                    if loi_ten:
                        return None, loi_ten
                if ten:
                    self.ns[ten] = ds
                    return f'{ten} = {ds}', None
                return str(ds), None

            gia_tri, loi_timeout = chay_co_timeout(fast_simplify, parsed)
            if loi_timeout:
                return None, loi_timeout
            loi_ten = self._kiem_tra_ten_la(gia_tri)
            if loi_ten:
                return None, loi_ten
            if ten:
                self.ns[ten] = gia_tri
                return f'{ten} = {gia_tri}', None
            return str(gia_tri), None
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

try:
    _CALC_POOL = mp.get_context('fork').Pool(1)
except ValueError:
    _CALC_POOL = None  # khong co fork (vd may local Windows) -> chay khong timeout


def chay_co_timeout(ham, *args):
    """Chay ham(*args) trong tien trinh con (fork, tao TRUOC CUDA nen an
    toan), gioi han TIMEOUT_SECONDS. Neu qua han, HUY va TAO LAI pool (vi
    tien trinh con cu van con chay ngam, khong the tai su dung duoc nua)
    roi tra ve loi ro rang thay vi treo may."""
    global _CALC_POOL
    if _CALC_POOL is None:
        return ham(*args), None
    ar = _CALC_POOL.apply_async(ham, args)
    try:
        return ar.get(timeout=TIMEOUT_SECONDS), None
    except mp.TimeoutError:
        _CALC_POOL.terminate()
        _CALC_POOL = mp.get_context('fork').Pool(1)
        return None, (f'computation timed out after {TIMEOUT_SECONDS}s '
                      '(the expression is too complex to simplify exactly). '
                      'Do not resend the exact same expression; try continuing '
                      'with a different, smaller step instead.')


tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
llm = LLM(model=MODEL, tensor_parallel_size=TENSOR_PARALLEL, dtype='float16',
          max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=0.90,
          trust_remote_code=True, enforce_eager=True, seed=SEED)
print('San sang.')

In [ ]:
# =====================================================================
# PROMPT (trich nguyen van tu KLTN_D37_ReAct_Calculator_1cau.ipynb)
# =====================================================================
# Doi sang dang khac => chi can viet lai PART_HUONGGIAI va PART_FEWSHOT,
# giu nguyen PART_TOOL va PART_KIENTHUC.

PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: t = -7/2 + 3/4*I` → `Observation: t = -7/2 + 3*I/4`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: mod_t = Abs(t)` then later `Action Input: k = -(11/6)/mod_t * t`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Imaginary unit is capital `I`; never lowercase `i`.
- Square root is `sqrt(...)`; never the `√` symbol.
- Power is `**` (e.g. `x**2`), not `^`.
- Plain fractions are already exact: writing `7/2` gives exactly seven-halves, never a decimal. You do NOT need any special function for fractions.
- Modulus of a complex number: `Abs(x)`. Conjugate: `conjugate(x)`.
- If a stored result is a LIST (e.g. from `solve(...)` with more than one solution), you can reference one element of it directly by index instead of retyping it: `name = solve(...)` then `name[0]` for the first element, `name[1]` for the second, and so on; use this the moment you need one specific element again in a later expression (e.g. `x_expr.subs(y, name[0])`). Retyping a long value from an `Observation` by hand (especially one with nested radicals) is exactly where a digit or a sign gets copied wrong; indexing the stored list can never have that problem, so prefer it every time.
- You can also build a list yourself, by hand, not only receive one from `solve(...)`: write `name = [val1, val2, val3, val4]` to store several numbers together in a single call, then index into it the same way (`name[0]`, `name[1]`, ...; see PART 4 for when to use this).
- To test whether two expressions are equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`. For most problem types this is decided by an exact symbolic proof. For this problem type specifically, it is decided by evaluating both sides to 50 significant decimal digits and checking they agree to that precision; this is not a formal proof, but for the kind of fixed numeric constants that appear in an exam's answer options, two genuinely different values could never coincide to 50 digits by chance, so treat a `True` here with the same confidence as an exact result. Precisely because it is a numeric check and not a symbolic proof, do not stop testing options as soon as one returns `True`; test all four, every time (see PART 4's matching step for why).
- To test a numeric inequality, write it directly with `<` or `>`, exactly like ordinary math notation: `a < b`, `a > b`, or even a chained double inequality in one call, `a < x < b`. The `Observation` will be `True` or `False`, decided the same way as `Eq()` (high-precision numeric comparison, not a symbolic proof). This works for any concrete numbers or stored names, including ones holding a nested radical.
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate anything yourself. Every equation in this problem type is at most quadratic (in $u$, after the substitution $u=t^2$), so `solve(...)` always returns an exact closed form; a numeric-only fallback is never needed here.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself. If the SAME error comes back again after you already tried to fix it once, that is a signal your whole APPROACH to this step is wrong, not just a small typo; retyping a small variation of the same idea will keep producing the same error. Stop and re-read PART 2's method for this exact step before trying again, rather than guessing another small variation of the same idea; never invent a placeholder word like `undefined` as if it were a value, since the Calculator has no such value, only real numbers, fractions, and radicals.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''


PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF COMPLEX NUMBERS**

1. **Definition**: a complex number is $z = a + bi$ with $a$ the real part, $b$ the imaginary part, and $i^2 = -1$.
2. **Operations**:
   - $(a+bi) \pm (c+di) = (a \pm c) + (b \pm d)i$
   - $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$
   - Multiplying by $i$ rotates by 90°: $i(a+bi) = -b + ai$.
3. **Conjugate**: $\overline{z} = a - bi$ when $z = a+bi$. Note $\overline{i} = -i$.
4. **Modulus**: $|z| = \sqrt{a^2+b^2}$. Note $|z| = |\overline{z}| = |-z|$, and $|i \cdot z| = |z|$.
5. **Triangle inequality**: $|z_1 + z_2| \ge |z_1| - |z_2|$.
   - **Equality condition**: equality holds if and only if $z_1$ and $z_2$ point in *opposite* directions, i.e. $z_1 = k z_2$ for a **real** number $k < 0$ (specifically $k = -|z_1|/|z_2|$).
6. **Triangle inequality, three terms**: $|z_1 + z_2 + z_3| \ge |z_1 + z_2| - |z_3| \ge |z_1| - |z_2| - |z_3|$.
   - **Equality condition**: equality holds if and only if $z_2$ and $z_3$ point in the *same* direction as each other, and both point *opposite* to $z_1$.
7. **Direction facts**: if $u$ points opposite to $v$, then $u = -\dfrac{|u|}{|v|} \cdot v$.
8. **Division**: to write $\dfrac{z_1}{z_2}$ in standard $a+bi$ form, multiply numerator and denominator by $\overline{z_2}$ (the conjugate of the denominator): $\dfrac{z_1}{z_2} = \dfrac{z_1\overline{z_2}}{z_2\overline{z_2}} = \dfrac{z_1\overline{z_2}}{|z_2|^2}$. Since $|z_2|^2$ is a positive real number, this removes $i$ from the denominator.
9. **Real/imaginary part notation**: for $z=a+bi$ ($a,b$ real), write $\mathrm{Re}(z)=a$ and $\mathrm{Im}(z)=b$.
10. **Pure imaginary number**: $z$ is pure imaginary if and only if $\mathrm{Re}(z)=0$ **and** $\mathrm{Im}(z)\ne0$ (the number $0$ itself is real, not pure imaginary, so $\mathrm{Im}(z)\ne0$ must be checked separately and never dropped).
11. **Modulus equation as an algebraic relation**: for $z=x+yi$ and a constant $z_0=x_0+y_0i$, the equation $|z-z_0|=R$ squares to $(x-x_0)^2+(y-y_0)^2=R^2$, which expands to a relation containing the cluster $x^2+y^2$ (plus linear terms in $x,y$ plus constants). Two such relations sharing the same $x^2+y^2$ cluster can be subtracted to eliminate it, leaving a linear equation in $x,y$.
12. **Modulus of a product**: $|u \cdot v| = |u| \cdot |v|$ for any complex numbers $u,v$ (this is why $|z^2-A|$ can be factored into $|z-w|\cdot|z+w|$ once $A=w^2$).
13. **Cauchy-Schwarz (B.C.S) inequality**: for real numbers $a,b,x,y$: $(ax+by)^2 \le (a^2+b^2)(x^2+y^2)$, with equality if and only if $(a,b)$ and $(x,y)$ are proportional. This turns an equation containing a mixed term like $cx+dy$ into a one-sided bound purely in terms of $x^2+y^2$.
14. **Square root of a complex number**: given $A=p+qi$, to find $w=c+di$ with $w^2=A$, expand $w^2=(c^2-d^2)+2cd\,i$ and match parts: $c^2-d^2=p$ and $2cd=q$. Squaring both and adding gives $(c^2+d^2)^2=p^2+q^2$, so $c^2+d^2=|A|$. Combined with $c^2-d^2=p$, this gives $c^2=\dfrac{|A|+p}{2}$ and $d^2=\dfrac{|A|-p}{2}$ (both always $\ge0$). Take $c=\sqrt{c^2}\ge0$; the sign of $d$ must match the sign of $q$ so that $2cd=q$ holds (if $q=0$, either sign works).
15. **Complex roots of a real-coefficient quadratic**: for $z^2+bz+c=0$ with $b,c$ real, write $\Delta'=(b/2)^2-c$. If $\Delta'<0$, the equation has no real root; its two roots are a complex-conjugate pair $z=-\dfrac{b}{2}\pm\sqrt{|\Delta'|}\,i$, where $|\Delta'|=-\Delta'$ since $\Delta'<0$ here.
16. **Conjugation as a geometric reflection**: the point representing $\overline{z}$ is the mirror image, across the real axis ($Ox$), of the point representing $z$. Consequently, if every vertex of a figure is replaced by its conjugate, the resulting figure is the mirror image of the original across $Ox$, so the two figures are congruent (in particular, they have equal area).
17. **Modulus of a difference as distance**: for $z_1=x_1+y_1i$ and $z_2=x_2+y_2i$, $|z_1-z_2|=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$ is exactly the distance between the points representing $z_1$ and $z_2$. This is why an equation like $|z-z_1|+|z-z_2|=K$ translates directly into $MA+MB=K$, where $M,A,B$ are the points representing $z,z_1,z_2$.
18. **Triangle inequality for three points**: for any three points $A,B,M$ in the plane, $MA+MB\ge AB$, with equality if and only if $M$ lies on the segment $AB$ (between $A$ and $B$, inclusive).
19. **The line through two points, and the case it cannot be written as $y=ax+b$.** For two distinct points $A(x_A,y_A)$ and $B(x_B,y_B)$: as a point moves from $A$ to $B$, its $x$-coordinate changes by $x_B-x_A$ and its $y$-coordinate changes by $y_B-y_A$. If $x_B\ne x_A$, the ratio $a=\dfrac{y_B-y_A}{x_B-x_A}$ (change in $y$ per unit change in $x$) is a genuine number, and the whole line is exactly the set of points $y=a(x-x_A)+y_A$. But if $x_B=x_A$, then $x$ never changes at all between $A$ and $B$; there is no \"change in $y$ per unit change in $x$\" to speak of, because $x$ does not change; so that ratio does not exist (computing it would divide by zero). This is not a computational accident; it is telling you something true about the line itself: when $x_A=x_B$, every point on the line through $A,B$ shares that same $x$-coordinate, so the line is exactly the vertical line $x=x_A$, and $y$ ranges freely over all values with no dependence on $x$ at all. So a line through two distinct points is always in exactly one of two situations: (i) $x_A\ne x_B$, describable as $y=a(x-x_A)+y_A$; or (ii) $x_A=x_B$, describable only as the vertical line $x=x_A$. These two situations are mirror images of each other with the roles of $x$ and $y$ swapped: whatever reasoning applies to \"$y$ as a function of $x$\" in case (i) has an exact analog, \"$x$ as a function of $y$\" (trivially constant, here), in case (ii).
20. **Vieta's formulas for a quadratic.** For $z^2+pz+q=0$ with roots $z_1,z_2$ (real or complex), the coefficients determine the roots' sum and product directly, without solving anything: $z_1+z_2=-p$ and $z_1z_2=q$. This holds whether the roots are real or a complex-conjugate pair (fact 15); the SAME two equations hold either way, since they simply come from matching coefficients in the identity $z^2+pz+q=(z-z_1)(z-z_2)=z^2-(z_1+z_2)z+z_1z_2$, which does not care whether $z_1,z_2$ happen to be real.
21. **Conjugate of a sum, and conjugate of a real multiple of $i$.** For any complex numbers $u,v$: $\overline{u+v}=\overline{u}+\overline{v}$ and $\overline{u-v}=\overline{u}-\overline{v}$ (conjugation distributes over addition/subtraction, since conjugating just flips the sign of every imaginary part, and imaginary parts add/subtract termwise). In particular, for a REAL constant $c$: $\overline{ci}=-ci$ (because $ci$ is purely imaginary with imaginary part $c$, flipping its sign gives $-ci$). Combining these: for any complex $z$ and real constant $c$, $\overline{z+ci}=\overline{z}+\overline{ci}=\overline{z}-ci$. This is the key trick that lets you replace $\overline{z}-ci$ by $\overline{z+ci}$ wherever it appears, which (by fact 4, $|\overline{w}|=|w|$) means $|\overline{z}-ci|=|z+ci|$; turning an expression that LOOKS like it needs both $z$ and $\overline{z}$ separately into one that depends only on the single quantity $z+ci$.
22. **Sum and difference of a complex number and its conjugate.** For $z=x+yi$ ($x,y$ real): combining fact 1 ($z=x+yi$) and fact 3 ($\overline{z}=x-yi$) directly by addition and subtraction gives $z+\overline{z}=2x$ and $z-\overline{z}=2yi$. This is the key move whenever a condition mixes $z$ and $\overline{z}$ through a sum or difference: $z+\overline{z}$ collapses to the single REAL number $2x$ (twice the real part, with no $y$ or $i$ left in it at all), and $z-\overline{z}$ collapses to the single PURELY IMAGINARY number $2yi$ (so $|z-\overline{z}|=2|y|$).
'''


PART_HUONGGIAI = r'''
**PART 2: HOW TO SOLVE THIS PROBLEM TYPE**

**Recognizing the shape, and the hidden link between its two constants.** The given equation always has the form $w|z|=\dfrac{K}{z}+(\text{a free complex constant})$, where $w=a+bi$ is the (constant, given) coefficient multiplying $|z|$ on the left, and $K$ is the (constant, given) numerator on the right. The free complex constant on the right is never independent of $w$: it is always EXACTLY $w \cdot i$. This is not a coincidence to verify each time; it is how this problem type is built, and it is the reason the equation can be factored at all. Multiplying $w$ by $i$ rotates it $90°$ (fact 2), turning $a+bi$ into $-b+ai$; recognizing that the free constant printed in the problem is precisely this rotated $w$ is what makes the next step possible.

**Factoring to isolate a single copy of $|z|-i$.** Since the free constant equals $w\cdot i$, the equation $w|z|=\dfrac{K}{z}+wi$ rearranges (moving $wi$ to the left) to $w|z|-wi=\dfrac{K}{z}$, and factoring $w$ out of the left side gives $w(|z|-i)=\dfrac{K}{z}$. Everything involving the unknown $z$ is now trapped inside two places only: the modulus $|z|$ (inside the factor $|z|-i$) and the lone $z$ in the denominator on the right.

**Taking the modulus of both sides.** By fact 12, $|uv|=|u|\cdot|v|$, and dividing works the same way for moduli: $\left|\dfrac{K}{z}\right|=\dfrac{|K|}{|z|}$. Applying this to $w(|z|-i)=\dfrac{K}{z}$ gives $|w|\cdot\big||z|-i\big|=\dfrac{|K|}{|z|}$. Every quantity here is now a plain nonnegative real number: $|w|$ and $|K|$ are fixed numbers computable directly from the problem's constants, and $|z|$ (which appears twice, once alone and once inside $\big||z|-i\big|$) is the one true unknown left.

**Introducing $t=|z|$, and evaluating $\big||z|-i\big|$ in terms of it.** Since $|z|$ is by definition a nonnegative real number, naming it $t=|z|$ turns the equation into one involving a single real unknown $t>0$ instead of the complex unknown $z$. The expression $|z|-i$ then becomes exactly $t-i$: a complex number with real part $t$ and imaginary part $-1$. By fact 4, its modulus is $\big|t-i\big|=\sqrt{t^2+(-1)^2}=\sqrt{t^2+1}$. Substituting this in gives a single equation purely in $t$: $|w|\cdot\sqrt{t^2+1}=\dfrac{|K|}{t}$.

**Clearing the square root and the fraction together.** Squaring both sides removes the square root ($|w|^2$ appears on the left, since $|w|$ was squared too; the right side becomes $\dfrac{K^2}{t^2}$, since $|K|^2=K^2$). Then multiplying both sides by $t^2$ (valid since $t>0$, so $t^2\ne0$) clears the fraction: $|w|^2\,t^2\,(t^2+1)=K^2$. Expanded, this contains only $t^4$ and $t^2$ (no odd power of $t$ at all): a biquadratic equation.

**Solving the biquadratic by substitution.** A biquadratic in $t$ is exactly a quadratic in disguise: setting $u=t^2$ (with the constraint $u>0$, inherited from $t>0$) turns $|w|^2t^4+|w|^2t^2-K^2=0$ into the ordinary quadratic $|w|^2u^2+|w|^2u-K^2=0$. By Vieta's formulas (fact 20, which holds for any quadratic regardless of the sign of its coefficients), the product of this quadratic's two roots is $\dfrac{-K^2}{|w|^2}$, a strictly negative number (since $K\ne0$ always here); a negative product means the two roots always have opposite signs, so exactly one of them is positive and the other is negative, every single time, with no case analysis needed beyond checking each root's own sign. Discard the negative root; the positive one is the true value of $u$.

**Recovering $|z|$ and reading off the answer.** Since $u=t^2$ and $t=|z|>0$, the answer is $t=\sqrt{u}$ (the positive square root; $t$ can never be negative, since it is a modulus). This number $t$ IS $|z|$: the exact value the four options are describing. The four options themselves come in different shapes: some state a single inequality ($|z|<c$ or $|z|>c$), some state a double inequality bracketing $|z|$ between two numbers ($c_1<|z|<c_2$), and some state an exact equality ($|z|=c$); reading which shape a given option has, and testing $t$ against exactly that shape, is the last step.
'''


PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (ReAct format)**

Đề bài: Xét số phức $z$ thỏa mãn $(1+2i)|z| = \dfrac{\sqrt{10}}{z}-2+i$. Mệnh đề nào dưới đây đúng?

A. $\dfrac{3}{2} < |z| < 2$.   B. $|z| > 2$.   C. $|z| < \dfrac{1}{2}$.   D. $\dfrac{1}{2} < |z| < \dfrac{3}{2}$.

<think>
Thought: The coefficient of $|z|$, quoted verbatim from "$(1+2i)|z|$", is $w=1+2i$. I store it.
Action: Calculator
Action Input: w = 1+2*I
Observation: w = 1 + 2*I

Thought: The numerator on the right, quoted verbatim from "$\dfrac{\sqrt{10}}{z}$", is $K=\sqrt{10}$. I store it.
Action: Calculator
Action Input: K = sqrt(10)
Observation: K = sqrt(10)

Thought: I check that the free constant matches PART 2's guarantee: I compute $w \cdot i$ and compare it against the free part read verbatim from the problem, $-2+i$.
Action: Calculator
Action Input: free_constant = w * I
Observation: free_constant = -2 + I

Thought: `free_constant` came back $-2+i$, matching the free part quoted verbatim from the problem exactly. So the equation really does factor as $w(|z|-i)=K/z$, and $|w|^2$ is what I need next, from fact 12's modulus-of-a-product step.
Action: Calculator
Action Input: w_abs_sq = Abs(w)**2
Observation: w_abs_sq = 5

Thought: The right side of the equation gets squared too, once both sides are squared; so I also need $K^2$, the other quantity PART 2's biquadratic requires.
Action: Calculator
Action Input: K_sq = K**2
Observation: K_sq = 10

Thought: Following PART 2's derivation, the biquadratic $|w|^2t^4+|w|^2t^2-K^2=0$ becomes, with $u=t^2$, the quadratic $|w|^2u^2+|w|^2u-K^2=0$. I solve it for $u$.
Action: Calculator
Action Input: u_sols = solve(Eq(w_abs_sq*u**2 + w_abs_sq*u - K_sq, 0), u)
Observation: u_sols = [-2, 1]

Thought: Two roots, as expected (opposite signs, by PART 2's Vieta argument). I check the sign of the first one, `u_sols[0]`.
Action: Calculator
Action Input: u_sols[0] > 0
Observation: False

Thought: `u_sols[0]` is not positive, so it is discarded. I check the second one, `u_sols[1]`.
Action: Calculator
Action Input: u_sols[1] > 0
Observation: True

Thought: `u_sols[1]` is positive; that is the valid $u$. I store it by name, using the index rather than retyping the number.
Action: Calculator
Action Input: u_val = u_sols[1]
Observation: u_val = 1

Thought: With $u$ known, $t=\sqrt{u}$ recovers the actual unknown by PART 2's substitution $u=t^2$; and $t$, in turn, IS $|z|$, so this is exactly the value every option is describing.
Action: Calculator
Action Input: t_val = sqrt(u_val)
Observation: t_val = 1

Thought: `t_val` is confirmed. Before testing anything, here are all four options copied fresh, side by side, exactly as printed: A: "$\dfrac{3}{2} < |z| < 2$."; B: "$|z| > 2$."; C: "$|z| < \dfrac{1}{2}$."; D: "$\dfrac{1}{2} < |z| < \dfrac{3}{2}$." I test them in this order, using exactly the symbol just copied for each one; I never approximate `t_val` to eyeball how close it looks to a number. Starting with A: two `<` symbols, $|z|$ between $\dfrac{3}{2}$ and $2$; the double-inequality shape.
Action: Calculator
Action Input: test_A = 3/2 < t_val < 2
Observation: test_A = False

Thought: B, copied above as "$|z| > 2$.": the symbol right after $|z|$ there is `>`.
Action: Calculator
Action Input: test_B = t_val > 2
Observation: test_B = False

Thought: C, copied above as "$|z| < \dfrac{1}{2}$.": the symbol right after $|z|$ there is `<`.
Action: Calculator
Action Input: test_C = t_val < 1/2
Observation: test_C = False

Thought: D, copied above as "$\dfrac{1}{2} < |z| < \dfrac{3}{2}$.": two `<` symbols, the double-inequality shape once more.
Action: Calculator
Action Input: test_D = 1/2 < t_val < 3/2
Observation: test_D = True

Thought: Before deciding, I check every result against the exact quotes copied at the start of this phase: A "$\dfrac{3}{2} < |z| < 2$." $\to$ `test_A=False`; B "$|z| > 2$." $\to$ `test_B=False`; C "$|z| < \dfrac{1}{2}$." $\to$ `test_C=False`; D "$\dfrac{1}{2} < |z| < \dfrac{3}{2}$." $\to$ `test_D=True`. Exactly one is `True`: `test_D`, matching the quote copied for D above; the letter is D.
</think>

Final Answer: \boxed{D}
'''


PART_NHIEMVU = r'''
**PART 4: YOUR TASK, STEP BY STEP**

The problem always has the form: $w|z|=\dfrac{K}{z}+(\text{free constant})$, with the free constant equal to $w\cdot i$; which one of four mentions about $|z|$ (an inequality, a double inequality, or an equality) is correct? Follow these steps, using the Calculator for every computation.

1. **Reading $w$ and $K$.** Quote the coefficient of $|z|$ on the left verbatim and store it as $w=a+bi$. Quote the numerator on the right (the constant divided by $z$) verbatim and store it as $K$.
2. **Confirming the factoring, and $|w|^2$, $K^2$.** Compute `w*I` with the Calculator (never by mental arithmetic, even though it looks simple) and compare the result against the free constant read verbatim from the problem; by PART 2, they always match, and this is what allows the equation to factor as $w(|z|-i)=K/z$. Then compute `w_abs_sq = Abs(w)**2` and `K_sq = K**2`. Never introduce `z` itself into a Calculator expression, here or anywhere else, to try to "verify" the original equation directly; `z` is never a declared free unknown in this problem type (only `u` is, from step 3 onward) and can never appear in any Calculator call; every quantity from here on is built purely from the known constants $w,K$ and the single unknown $u=t^2$.
3. **Solving the biquadratic via $u=t^2$.** Compute `u_sols = solve(Eq(w_abs_sq*u**2 + w_abs_sq*u - K_sq, 0), u)`. This always returns exactly two roots.
4. **Picking the positive root.** For each element of `u_sols` in turn, test its sign with `u_sols[<index>] > 0`. By PART 2's Vieta argument, exactly one element will test `True`; store that one (by indexing, never by retyping its value) as `u_val`.
5. **Recovering $|z|$.** Compute `t_val = sqrt(u_val)`. This is exactly $|z|$.
6. **Copying all four options fresh, together, before testing any of them.** Do not assume any option's shape or direction from the worked example; a different problem can pair completely different directions with the letters A, B, C, D. In the Thought that introduces your first test, copy all four options' exact text side by side (A: "...", B: "...", C: "...", D: "..."), reading each one character by character, with particular attention to whether its symbol is `<`, `>`, or `=`. This copied set is what every test below is checked against.
7. **Testing each option, storing each result under its own name.** For each option A, B, C, D in turn, using the shape identified from the text copied in step 6 (a single inequality $|z|<c$ or $|z|>c$, a double inequality $c_1<|z|<c_2$, or an equality $|z|=c$): compute it as a NAMED result, `test_A = <matching expression>` (likewise `test_B`, `test_C`, `test_D`), using `t_val < c`, `t_val > c`, `c_1 < t_val < c_2`, or `Eq(t_val, c)`, keeping the `<`/`>` direction exactly as copied. Do this for all four, even after one already returns `True`; this numeric check is not a formal proof, and never approximate `t_val` (e.g. with `.evalf()`) to eyeball how close it looks to an option's number.
8. **Final cross-check against the copied text, before deciding.** In one closing Thought, line up each option's text copied in step 6 next to its stored result (A → `test_A`, B → `test_B`, C → `test_C`, D → `test_D`). Exactly one should be `True`; that letter is the answer. If that is not what this side-by-side check shows, some test's expression does not actually match its copied text; find and fix that mismatch and recompute before deciding. Write the final line `Final Answer: \boxed{<Letter>}` using that letter.

Đề bài:
{de_bai}
'''


PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)

In [ ]:
# =====================================================================
# VONG LAP ReAct THEO LO (batch) - khac biet duy nhat so voi ban 1 cau
# (MayTinh/fast_simplify/is_zero da duoc dinh nghia o Cell 3, truoc khi
# LLM(...) khoi tao - xem giai thich an toan CUDA/fork o do)
# =====================================================================
STOP_STR = 'Observation:'
MAX_TOOL_CALLS = 26          # quy trinh day du ~9 luot tinh (w,K,w_abs_sq,K_sq,u_sols,2x kiem tra dau,u_val,t_val) + toi da 4 luot doi chieu phuong an, cong du cho vai lan sua loi cu phap
ACTION_INPUT_RE = re.compile(r'Action Input:[ \t]*(.*)')
FINAL_ANSWER_RE = re.compile(r'Final\s+Answer\s*:\s*\\boxed\{\s*[A-D]\s*\}')


class TrangThai:
    """Trang thai ReAct rieng cua 1 cau (bo nho bien rieng, ngan sach rieng)."""

    def __init__(self, rec, chat_prompt):
        self.rec = rec
        self.chat_prompt = chat_prompt
        self.may_tinh = MayTinh(an_so_tu_do=AN_SO_TU_DO)   # bo nho + an so RIENG cho tung cau
        self.full_text = '<think>\nThought:'
        self.tong_tok = 0
        self.n_calls = 0
        self.nhat_ky = []
        self.xong = False
        self.ly_do_dung = ''
        self.buoc_chot = False             # het luot tool -> ep viet Final Answer


prompts_ban_dau = []
for r in records:
    prompt_text = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts_ban_dau.append(tok.apply_chat_template(
        [{'role': 'user', 'content': prompt_text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

ds = [TrangThai(r, p) for r, p in zip(records, prompts_ban_dau)]

t0 = time.time()
vong = 0
while True:
    hoat_dong = [s for s in ds if not s.xong]
    if not hoat_dong:
        break
    vong += 1

    lo = []
    for s in hoat_dong:
        dau_vao = s.chat_prompt + s.full_text
        cho_trong = MAX_MODEL_LEN - len(tok(dau_vao).input_ids) - 8
        so_sinh = min(MAX_NEW_TOKENS - s.tong_tok, cho_trong)
        if so_sinh <= 0:
            s.xong = True
            s.ly_do_dung = 'het_cho_context' if cho_trong <= 0 else 'het_ngan_sach_token'
            continue
        stop = None if s.buoc_chot else [STOP_STR]
        lo.append((s, dau_vao, SamplingParams(
            temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
            presence_penalty=PRESENCE_PENALTY, max_tokens=so_sinh,
            seed=SEED, stop=stop)))

    if not lo:
        break

    outs = llm.generate([p for _, p, _ in lo],
                        [sp for _, _, sp in lo], use_tqdm=False)

    n_goi_vong_nay = 0
    for (s, _, _), out in zip(lo, outs):
        o = out.outputs[0]
        s.full_text += o.text
        s.tong_tok += len(o.token_ids)

        if s.buoc_chot:
            s.xong = True
            s.ly_do_dung = 'het_han_muc_tool'
            continue

        m_final = FINAL_ANSWER_RE.search(s.full_text)
        if m_final:
            s.full_text = s.full_text[:m_final.end()]
            s.xong = True
            s.ly_do_dung = 'model_ket_thuc'
            continue

        if o.stop_reason != STOP_STR:
            s.xong = True
            s.ly_do_dung = ('model_ket_thuc' if o.finish_reason == 'stop'
                            else 'het_token')
            continue

        s.n_calls += 1
        n_goi_vong_nay += 1
        # D37-only fix: khi 1 luot generate() lo sinh QUA 1 khoi
        # Thought/Action/Action-Input (stop=['Observation:'] chi kich hoat
        # khi CHINH MODEL tu viet chu do, nen no co the ramble sang khoi
        # thu 2 truoc khi dung), luon lay Action Input DAU TIEN (dung tinh
        # than ReAct - moi luot chi 1 action) va cat bo phan sinh them sau
        # do khoi s.full_text, thay vi am tham lay Action Input CUOI CUNG
        # va bo qua dong dau (day la nguyen nhan loi "undefined name" quan
        # sat duoc o STT776).
        cac_khop = list(ACTION_INPUT_RE.finditer(o.text))
        khop = cac_khop[0] if cac_khop else None
        bieu_thuc = khop.group(1).strip() if khop else ''
        if len(cac_khop) > 1:
            vi_tri_cat = len(s.full_text) - len(o.text) + khop.end()
            s.full_text = s.full_text[:vi_tri_cat]

        if not bieu_thuc:
            quan_sat = ('ERROR: no "Action Input:" line found. Write a Thought, '
                        'then "Action: Calculator", then "Action Input: <expression>".')
            s.nhat_ky.append(('(khong co Action Input)', quan_sat))
        else:
            ket_qua, loi = s.may_tinh.tinh(bieu_thuc)
            if loi:
                quan_sat = (f'ERROR: {loi}. Fix the syntax and try again '
                            '(use I for the imaginary unit, sqrt() for roots, '
                            '** for powers, Abs()/conjugate(), Eq(a,b) to compare).')
                s.nhat_ky.append((bieu_thuc, f'ERROR: {loi}'))
            else:
                quan_sat = ket_qua
                s.nhat_ky.append((bieu_thuc, ket_qua))

        s.full_text += f'{STOP_STR} {quan_sat}\n'

        if s.n_calls >= MAX_TOOL_CALLS:
            s.full_text += ('\n[SYSTEM: tool-call limit reached - give your Final '
                            'Answer now, with no further Action.]\n')
            s.buoc_chot = True

    print(f'  Vong {vong:2d}: {len(lo):2d} cau sinh, {n_goi_vong_nay:2d} luot goi tool, '
          f'con lai {sum(1 for s in ds if not s.xong):2d} cau '
          f'({time.time()-t0:.0f}s)')

print(f'Xong toan bo sau {vong} vong, {(time.time()-t0)/60:.1f} phut.')


# =====================================================================
# CHAM DIEM + LUU KET QUA
# =====================================================================
def lay_dap_an(t):
    for pat in (r'Final\s+Answer[^A-D]{0,20}([A-D])\b',
                r'\\boxed\{\s*([A-D])\s*\}',
                r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b'):
        m = re.findall(pat, t)
        if m:
            return m[-1]
    return ''


rows = []
for s in ds:
    chon = lay_dap_an(s.full_text)
    n_loi = sum(1 for _, obs in s.nhat_ky if str(obs).startswith('ERROR'))
    rows.append({
        'STT': s.rec['STT'],
        'loai_so': s.rec.get('loai_so', ''),
        'dap_an_dung': s.rec['dap_an_letter'],
        'model_chon': chon,
        'DUNG': chon == s.rec['dap_an_letter'],
        'token': s.tong_tok,
        'so_luot_goi_tool': s.n_calls,
        'so_loi_cu_phap': n_loi,
        'ly_do_dung': s.ly_do_dung,
        'nhat_ky_tool': ' | '.join(f'{bt} -> {obs}' for bt, obs in s.nhat_ky),
        'full_text': s.full_text,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('=' * 72)
print(f"KET QUA: {df['DUNG'].sum()} / {len(df)} DUNG ({df['DUNG'].mean()*100:.1f}%)")
print('=' * 72)
print()
print('--- Theo loai so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Chat luong dung tool ---')
print(f"Trung binh luot goi tool/cau : {df['so_luot_goi_tool'].mean():.1f}")
print(f"Tong luot goi tool           : {df['so_luot_goi_tool'].sum()}")
print(f"Tong luot LOI cu phap        : {df['so_loi_cu_phap'].sum()} "
      f"({df['so_loi_cu_phap'].sum() / max(1, df['so_luot_goi_tool'].sum()) * 100:.1f}%)")
print(f"So cau KHONG goi tool lan nao: {(df['so_luot_goi_tool'] == 0).sum()}")
print(f"Trung binh token/cau         : {df['token'].mean():.0f}")
print()
print('--- Ly do dung ---')
print(df['ly_do_dung'].value_counts())
print()
if (~df['DUNG']).any():
    print('--- CAC CAU SAI ---')
    print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon',
                           'token', 'so_luot_goi_tool', 'so_loi_cu_phap',
                           'ly_do_dung']].to_string(index=False))
else:
    print(f"*** TAT CA {len(df)} CAU DEU DUNG ***")
print()
print('Da luu:', OUT_PATH)